# 7. Transit and radial-velocity validation

Two independent observational pipelines, each validated against known
answers before being trusted on a candidate: transit depths fitted from
synthetic and real light curves, and the reliability gate that stops a
radial-velocity fit from fabricating a planet mass out of noise.


This notebook is part of the reproducibility set for **Finding Earth 2.0 in
Distant Worlds**. It reads the same committed data every other output in this
project reads (`results/`, `data/processed/`, `data/manifests/`) and calls
the same `earth2` functions the pipeline itself calls -- nothing here is a
simplified restatement computed a different way. Run `python -m earth2 all`
first if `results/` does not exist yet.

See `docs/METHODS.md` for the full equations and `docs/LIMITATIONS.md` for
this project's stated caveats.


In [1]:
import sys
sys.path.insert(0, "../src")

import json
import numpy as np
import matplotlib.pyplot as plt

from earth2.config import ROOT
from earth2.transit.lightcurve import fit_trapezoid, bls_period_search


## Transit fitting on a synthetic, known-answer transit

Before trusting a fit on real, noisy data, confirm it recovers a transit
whose depth and duration are known exactly.


In [2]:
rng = np.random.default_rng(20260824)
true_depth_ppm, true_duration_hours = 8000.0, 2.4
phase = np.linspace(-0.6, 0.6, 500)
flux = np.ones_like(phase)
in_transit = np.abs(phase) < (true_duration_hours / 24.0) / 2.0
flux[in_transit] -= true_depth_ppm / 1e6
flux += rng.normal(0, 0.00025, len(phase))

fit = fit_trapezoid(phase, flux, duration_guess_hours=true_duration_hours)
print(f"True depth:   {true_depth_ppm:.0f} ppm   Fitted: {fit.depth_ppm:.0f} ppm")
print(f"True duration: {true_duration_hours:.2f} h   Fitted: {fit.duration_hours:.2f} h")
print(f"Significant (SNR>=7)? {fit.significant}   (SNR={fit.depth_snr:.1f})")


True depth:   8000 ppm   Fitted: 7978 ppm
True duration: 2.40 h   Fitted: 2.49 h
Significant (SNR>=7)? True   (SNR=200.7)


## Real benchmark: four bright, previously-known transiting planets

`python -m earth2 validate-transit` runs this pipeline against public MAST
light curves for planets whose depths are independently published, and
records the ratio of fitted-to-published depth. TRAPPIST-1 itself is
deliberately **not** validated this way -- see the note below.


In [3]:
val = json.loads((ROOT / "results" / "transit_validation.json").read_text())
print(val["purpose"])
print()
for t in val["targets"]:
    status = "VALIDATED" if t.get("validated") else t["status"].upper()
    ratio = t.get("ratio_fitted_to_published")
    ratio_s = f"{ratio:.2f}" if ratio is not None else "--"
    print(f"{t['planet']:16s} published={t.get('published_depth_ppm', 0):>7.0f} ppm  "
          f"fitted={t.get('fitted_depth_ppm', 0) or 0:>7.0f} ppm  ratio={ratio_s}  {status}")
print()
print(f"Median fitted/published ratio: {val['median_ratio_fitted_to_published']:.2f}")
print(val["systematic_note"])


Validation of the transit pipeline against planets whose depths are independently known. These are bright hot Jupiters and sub-Neptunes chosen for signal strength, NOT Earth-2.0 candidates, and they are excluded from the candidate ranking.

HD 209458 b      published=  15000 ppm  fitted=  11555 ppm  ratio=0.77  VALIDATED
HD 189733 b      published=  24000 ppm  fitted=  22659 ppm  ratio=0.94  VALIDATED
WASP-39 b        published=  23435 ppm  fitted=  18790 ppm  ratio=0.80  VALIDATED
WASP-19 b        published=  20770 ppm  fitted=  20664 ppm  ratio=0.99  VALIDATED
GJ 1214 b        published=  13430 ppm  fitted=      0 ppm  ratio=--  NO_DATA

Median fitted/published ratio: 0.87
Fitted depths run consistently below published values. This is a known systematic of the Savitzky-Golay detrending step, which absorbs a little of the transit even with its window forced to at least three times the transit duration. Depths from this pipeline are approximate and biased slightly low; published limb-d

## Why TRAPPIST-1 does not get a transit fit here

At Tmag 14.9 its per-cadence photometric precision (~33,000 ppm) is several
times its transit depth (~7,000 ppm); only one TESS sector is public; seven
planets transit the same star; and large transit-timing variations mean a
fold on any single planet's period smears the other six through it. Every
fit attempted there disagreed with the published depth by a factor of 2-4 --
reported as `fit_not_validated`, not tuned until it looked convincing.


## The RV reliability gate: an ungated fit fabricates a mass

Fitting a fixed-period circular orbit to too few radial-velocity points can
return a *confident-looking* amplitude from pure noise. This happened for
real, on this exact target, before the gate existed (see
`docs/RESEARCH_NOTES.md`, "An ungated RV amplitude fit fabricated a planet
mass").


In [4]:
from earth2.radial_velocity.rv import _fit_sinusoid

# Too few points, pure noise -- the historical TRAPPIST-1 failure mode.
rng2 = np.random.default_rng(2)
t_sparse = np.sort(rng2.uniform(0, 100, 5))
y_noise = rng2.normal(0, 20, len(t_sparse))
bad_fit = _fit_sinusoid(t_sparse, y_noise, None, period=3.15)
print("5 noisy points, forced fit:")
print(f"  Fitted semi-amplitude: {bad_fit['semi_amplitude_ms']:.1f} m/s")
print(f"  Reliable? {bad_fit['reliable']}")
print(f"  Reasons: {bad_fit['unreliable_because']}")


5 noisy points, forced fit:
  Fitted semi-amplitude: 3.4 m/s
  Reliable? False
  Reasons: ['fewer than 20 velocities (5)', 'amplitude significance below 3 sigma (K/sigma_K = 0.2)']


In [5]:
# The same fit on a genuine signal with enough points passes cleanly.
t_good = np.sort(rng2.uniform(0, 200, 40))
y_good = 8.0 * np.sin(2 * np.pi * t_good / 12.0) + rng2.normal(0, 0.4, len(t_good))
good_fit = _fit_sinusoid(t_good, y_good, None, period=12.0)
print(f"40 points, real 8 m/s signal -> fitted {good_fit['semi_amplitude_ms']:.1f} m/s, "
      f"reliable={good_fit['reliable']}")


40 points, real 8 m/s signal -> fitted 8.0 m/s, reliable=True
